# ZAIC 2026 LifeMate — v4: fine-tune Whisper + nộp public lần 2

v3 cho thấy ASR làm mất **một nửa** điểm (val_unseen: transcript gold 49.0 → Whisper 25.3). Vòng này fine-tune whisper-large-v3-turbo trên 5036 clip của train (giữ lại clip của val để đo), LLM giữ nguyên adapter v2.

Upload `lifemate_baseline.zip` **bản mới** vào `MyDrive/ZAIC/` → **Run all** (L4 khoảng 1.5 giờ) → nộp `submission_public_v4.jsonl` (chọn **Based on small models (<6B)**) → gửi mình `results_v4.zip` + điểm leaderboard.

In [ ]:
# ==== CẤU HÌNH ====
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ASR_BASE   = "openai/whisper-large-v3-turbo"
ASR_EPOCHS = 3
BATCH      = 32
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ADAPTER    = DRIVE_DIR + "/ft_fit_adapter"
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", "/content/results_v4"
LLM, ASR_FT = "/content/ft_fit/merged", "/content/asr_ft"

In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
assert os.path.exists(f"{WORK}/lifemate/asr_finetune.py"), "lifemate_baseline.zip trong Drive là bản cũ -> upload bản mới"
assert os.path.exists(ADAPTER + "/adapter_config.json"), "Không thấy adapter v2 trong Drive: " + ADAPTER
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA:", DATA)

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-1000:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt

## 1. Fine-tune Whisper (giữ lại clip của val). Theo dõi `loss` / `eta`

In [ ]:
%cd {WORK}
!python -m lifemate.asr_finetune --root {DATA} --exclude splits/val_all.txt --model {ASR_BASE} --out {ASR_FT} --epochs {ASR_EPOCHS} 2>&1 | grep -vi warn
assert os.path.exists(f"{ASR_FT}/config.json"), "ASR fine-tune lỗi: xem log phía trên"
shutil.copy(f"{ASR_FT}/asr_train_log.txt", f"{RES}/asr_train_log.txt")
shutil.copytree(ASR_FT, os.path.join(DRIVE_DIR, "asr_ft_v4"), dirs_exist_ok=True); print("ASR saved to Drive")

## 2. WER trên clip val (so với v3: turbo gốc WER 34.6%)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split train --ids splits/val_all.txt --model {ASR_FT} --dtype {DTYPE} --out asr_ft_val.json 2>&1 | grep -vi warn | tail -4
shutil.copy("asr_ft_val.json.wer.txt", f"{RES}/wer_asr_ft.txt")

## 3. Điểm end-to-end trên val (v3: gold 49.0 / 67.7, turbo gốc 25.3 / 35.6)

In [ ]:
%cd {WORK}
!python -m lifemate.merge --base {BASE_MODEL} --adapter {ADAPTER} --out {LLM} 2>&1 | grep -vi warn | tail -1
sc = {}
for S in ["val_unseen", "val_seen"]:
    !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts asr_ft_val.json --out runs/asrft_{S} 2>&1 | tail -1
    !python scripts/score.py --root {DATA} --split train --pred runs/asrft_{S}/submission.jsonl --by-tag > {RES}/score_asrft_{S}.txt
    !python -m lifemate.report --root {DATA} --split train --pred runs/asrft_{S}/submission.jsonl --show 6 > {RES}/report_asrft_{S}.txt
    sc[S] = float(open(f"{RES}/score_asrft_{S}.txt").readline().split()[-1])
print(sc); json.dump(sc, open(f"{RES}/e2e_asrft.json", "w"))

## 4. Public → `submission_public_v4.jsonl`

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split public_test --model {ASR_FT} --dtype {DTYPE} --out asr_public_v4.json 2>&1 | grep -vi warn | tail -2
!python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts asr_public_v4.json --out runs/public_v4 2>&1 | tail -1
for src, dst in [("runs/public_v4/submission.jsonl", "submission_public_v4.jsonl"), ("runs/public_v4/traces.jsonl", "traces_public_v4.jsonl"), ("asr_public_v4.json", "asr_public_v4.json")]:
    shutil.copy(src, f"{RES}/{dst}")
shutil.copy("runs/public_v4/submission.jsonl", os.path.join(DRIVE_DIR, "submission_public_v4.jsonl"))
%cd {DATA}/public_test
!python -m zaic_tools --split public_test --root .. --validate {RES}/submission_public_v4.jsonl 2>&1 | tail -3 | tee {RES}/validate_public_v4.txt

## 5. Đóng gói → `results_v4.zip`

In [ ]:
open(f"{RES}/config.txt", "w").write(f"LLM=v2 adapter ft_fit\nASR={ASR_BASE} fine-tuned {ASR_EPOCHS} ep\nGPU={GPU_NAME}\n")
z = shutil.make_archive("/content/results_v4", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results_v4.zip"))
print("saved", os.path.join(DRIVE_DIR, "results_v4.zip"))
from google.colab import files
files.download(z); files.download(f"{RES}/submission_public_v4.jsonl")